# Download speed benchmark — CT-RATE / HuggingFace

Standalone, throwaway test. Compares three ways of downloading `.nii.gz` volumes
from the CT-RATE dataset to find what's actually limiting throughput in
`extract_ctclip_features.ipynb`, whose Cell 6 currently downloads one file at a
time via `HfFileSystem` (~8s/volume, ~72% of total per-volume time there).

**Run this in a SEPARATE Colab runtime from the live extraction notebook.**
It does not touch rclone, Google Drive, the checkpoint, or the GPU — it only
downloads a small sample to `/content/bench_tmp`, times it, and deletes it
again. Safe to run in parallel, but a runtime of its own avoids two processes
competing for the same network link.

You'll need `HF_TOKEN` available the same way as the main notebook (a Colab
Secret with 'Notebook access' toggled on for **this** notebook too — secrets
don't automatically carry over — or a real env var outside Colab).

In [ ]:
# ── CELL 1 — Setup ──────────────────────────────────────────────────────
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass

HF_TOKEN = "YOUR_HF_TOKEN"
assert HF_TOKEN, (
    "No HF_TOKEN. Add it as a notebook Secret (key icon, left sidebar) with "
    "'Notebook access' on for THIS notebook, or export HF_TOKEN outside Colab. "
    "It must have accepted the CT-RATE gated-dataset terms."
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "hf_transfer"],
    check=True,
)

HF_REPO = "ibrahimhamamci/CT-RATE"
SPLIT_ROOT = "dataset/train_fixed"
N_SAMPLE = 24  # ~4GB total at the ~164MB/volume average from the main notebook's discovery step
BENCH_DIR = Path("/content/bench_tmp")
BENCH_DIR.mkdir(parents=True, exist_ok=True)


def clear_bench_dir() -> None:
    """hf_hub_download recreates the repo's folder structure under BENCH_DIR,
    so a flat glob().unlink() wouldn't catch it - just nuke and recreate."""
    if BENCH_DIR.exists():
        shutil.rmtree(BENCH_DIR)
    BENCH_DIR.mkdir(parents=True, exist_ok=True)


print("Setup complete.")

In [ ]:
# ── CELL 2 — Sample a handful of volume paths ──────────────────────────
# Stops after N_SAMPLE instead of walking the full ~47k-entry tree like the
# main notebook's discovery step does - this is just a quick throwaway sample.
from huggingface_hub import HfApi

api = HfApi()
sample_paths = []
for entry in api.list_repo_tree(
    HF_REPO, repo_type="dataset", path_in_repo=SPLIT_ROOT,
    recursive=True, token=HF_TOKEN,
):
    path = getattr(entry, "path", "")
    if path.endswith(".nii.gz"):
        sample_paths.append(path)
    if len(sample_paths) >= N_SAMPLE:
        break

print(f"Sampled {len(sample_paths)} volume(s) for benchmarking.")

In [ ]:
# ── CELL 3 — Benchmark A: sequential (current approach in extract_ctclip_features.ipynb) ──
from huggingface_hub import HfFileSystem


def bench_sequential(paths):
    fs = HfFileSystem(token=HF_TOKEN)
    clear_bench_dir()
    t0 = time.perf_counter()
    total_bytes = 0
    for i, hf_path in enumerate(paths):
        out = BENCH_DIR / f"seq_{i}.nii.gz"
        with fs.open(f"datasets/{HF_REPO}/{hf_path}", "rb") as src, open(out, "wb") as dst:
            data = src.read()
            dst.write(data)
            total_bytes += len(data)
    elapsed = time.perf_counter() - t0
    clear_bench_dir()
    return elapsed, total_bytes


elapsed_seq, bytes_seq = bench_sequential(sample_paths)
rate_seq = len(sample_paths) / elapsed_seq * 3600
print(
    f"SEQUENTIAL: {elapsed_seq:.1f}s for {len(sample_paths)} volumes "
    f"({elapsed_seq / len(sample_paths):.2f}s/vol, {rate_seq:.0f} vol/h, "
    f"{bytes_seq / 1e6 / elapsed_seq:.1f} MB/s)"
)

In [ ]:
# ── CELL 4 — Benchmark B: threaded, same HfFileSystem, no hf_transfer ─────
from concurrent.futures import ThreadPoolExecutor


def download_one(fs, hf_path, out_path):
    with fs.open(f"datasets/{HF_REPO}/{hf_path}", "rb") as src, open(out_path, "wb") as dst:
        data = src.read()
        dst.write(data)
        return len(data)


def bench_threaded(paths, max_workers):
    fs = HfFileSystem(token=HF_TOKEN)
    clear_bench_dir()
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        futures = [
            ex.submit(download_one, fs, p, BENCH_DIR / f"thr_{i}.nii.gz")
            for i, p in enumerate(paths)
        ]
        total_bytes = sum(f.result() for f in futures)
    elapsed = time.perf_counter() - t0
    clear_bench_dir()
    return elapsed, total_bytes


for workers in (4, 8, 16):
    elapsed, total_bytes = bench_threaded(sample_paths, workers)
    rate = len(sample_paths) / elapsed * 3600
    print(
        f"THREADED(workers={workers}): {elapsed:.1f}s "
        f"({elapsed / len(sample_paths):.2f}s/vol, {rate:.0f} vol/h, "
        f"{total_bytes / 1e6 / elapsed:.1f} MB/s)"
    )

In [ ]:
# ── CELL 5 — Benchmark C: hf_transfer (Rust, multi-connection chunked transfer) ──
import importlib

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

import huggingface_hub

importlib.reload(huggingface_hub)  # pick up the env var if the module was already imported
from huggingface_hub import hf_hub_download


def bench_hf_transfer(paths):
    clear_bench_dir()
    t0 = time.perf_counter()
    total_bytes = 0
    for p in paths:
        local = hf_hub_download(
            repo_id=HF_REPO, repo_type="dataset", filename=p,
            token=HF_TOKEN, local_dir=str(BENCH_DIR),
        )
        total_bytes += Path(local).stat().st_size
    elapsed = time.perf_counter() - t0
    clear_bench_dir()
    return elapsed, total_bytes


elapsed_ht, bytes_ht = bench_hf_transfer(sample_paths)
rate_ht = len(sample_paths) / elapsed_ht * 3600
print(
    f"HF_TRANSFER (sequential): {elapsed_ht:.1f}s "
    f"({elapsed_ht / len(sample_paths):.2f}s/vol, {rate_ht:.0f} vol/h, "
    f"{bytes_ht / 1e6 / elapsed_ht:.1f} MB/s)"
)

In [ ]:
# ── CELL 6 — How to read the results ──────────────────────────────────
print("=" * 70)
print("Compare the vol/h and MB/s numbers printed above:")
print("- THREADED >> SEQUENTIAL  -> downloads were latency-bound (waiting on")
print("  request/response, not raw bandwidth). Apply threading to Cell 6's")
print("  download step in extract_ctclip_features.ipynb.")
print("- HF_TRANSFER >> SEQUENTIAL/THREADED -> a single connection per file was")
print("  bandwidth-capped; the Rust multi-connection transfer helps more.")
print("  Consider switching the main notebook to hf_hub_download with")
print("  HF_HUB_ENABLE_HF_TRANSFER=1 instead of HfFileSystem.")
print("- If both help, combining threaded + hf_transfer is worth trying too.")
print("=" * 70)